# Train the Khmer OCR recognizer

Works unmodified on **Colab**, **Kaggle**, and a **local machine** -- on **CPU, GPU, or TPU**:
- Colab: mounts Google Drive at `/content/drive/My Drive/tuna-ocr` and checkpoints there.
- Kaggle: checkpoints to `/kaggle/working/tuna-ocr` (persisted as notebook output).
- Local: checkpoints to `recognizer/checkpoints/` in the repo.

The accelerator (TPU / GPU / CPU) is auto-detected -- no notebook changes needed either
way. **To actually get a TPU**, select it as the runtime/accelerator in Colab
(Runtime > Change runtime type > TPU) or Kaggle (Settings > Accelerator > TPU) *before*
running this notebook; both platforms ship `torch_xla` preinstalled on their TPU
runtimes, so no extra install step is needed here. On GPU, batch size is auto-probed
to fit the available VRAM (OOM-probing auto-tune); on TPU that probe is skipped (XLA
doesn't surface a catchable Python OOM the same way) and the configured batch size is
used as-is.

Training logs every 100 steps and pushes a checkpoint to the `Panhapich/tuna-ocr`
Hugging Face repo (created private) every 10,000 steps.

**Before running:** add an `HF_TOKEN` secret (Colab: key icon in the left sidebar;
Kaggle: Add-ons > Secrets; local: `export HF_TOKEN=hf_...`).

**TPU caveat:** PyTorch/XLA's support for the CTC loss op has historically been
inconsistent across versions -- if training errors out or looks unusually slow on TPU,
check whether `torch.nn.functional.ctc_loss` is silently falling back to a CPU path
before assuming it's a bug in this repo.

In [ ]:
import os, subprocess, sys

def detect_environment():
    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working"):
        return "kaggle"
    return "local"

ENV = detect_environment()
print("environment:", ENV)

REPO_URL = "https://github.com/Pich09/tuna-ocr.git"

if not os.path.isdir("recognizer"):
    subprocess.run(["git", "clone", REPO_URL, "tuna-ocr"], check=True)
    os.chdir("tuna-ocr")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

In [ ]:
!pip install -q -r recognizer/requirements.txt -r real_data/requirements.txt

In [ ]:
from recognizer import env_utils

checkpoint_root = env_utils.get_checkpoint_root(ENV)
hf_token = env_utils.get_hf_token(ENV)
accelerator = env_utils.detect_accelerator()
print("checkpoint root:", checkpoint_root)
print("HF token loaded:", bool(hf_token))
print("accelerator:", accelerator)
if accelerator == "cpu":
    print("no GPU/TPU detected -- training will be slow. On Colab: Runtime > Change "
          "runtime type. On Kaggle: Settings > Accelerator.")

In [ ]:
# Downloads Panhapich/khmer-sp-8k's SentencePiece model + khmer_segmentation.py
# wrapper (a bare .model file is not enough -- see recognizer/README.md).
from recognizer.tokenizer.fetch_tokenizer import fetch_tokenizer

fetch_tokenizer()

In [ ]:
# Pull training samples from the 4 configured real_data sources, if not already present.
from pathlib import Path
from real_data.config import EXTERNAL_DATASETS, REAL_DATA_ROOT

NUM_SAMPLES_PER_SOURCE = 2000  # raise for a real training run

real_data_roots = []
for source in EXTERNAL_DATASETS:
    out_dir = REAL_DATA_ROOT / "samples" / source
    real_data_roots.append(out_dir)
    if (out_dir / "manifest.tsv").exists():
        print(f"{source}: already present, skipping")
        continue
    print(f"{source}: pulling {NUM_SAMPLES_PER_SOURCE} samples...")
    !python -m real_data.generate_external_chunks --source {source} --num-samples {NUM_SAMPLES_PER_SOURCE}

print(real_data_roots)

In [ ]:
# Deduplicate across all pulled sources before training -- catches
# exact/near-duplicate images (e.g. two sources that happen to overlap),
# so training never sees the same line image twice. Writes a pooled,
# deduplicated manifest + a JSON report (real_data/samples/dedup/).
dedup_manifest = REAL_DATA_ROOT / "samples" / "dedup" / "manifest.tsv"
real_data_dirs_str = " ".join(str(p) for p in real_data_roots)
!python -m real_data.deduplicate --real-data-dirs {real_data_dirs_str} --out-dir {dedup_manifest.parent}

In [ ]:
from recognizer.config import ModelConfig, TrainConfig
from recognizer.train import run_training

model_cfg = ModelConfig()
train_cfg = TrainConfig()  # log_every=100, ckpt_every=10_000 by default

# The Panhapich/tuna-ocr HF repo is created private by default the first time
# a checkpoint is pushed (hub_private=True) -- set to False only if you've
# deliberately decided the checkpoint repo should be public.
model = run_training(
    model_cfg, train_cfg,
    dedup_manifest_path=dedup_manifest,
    checkpoint_root=checkpoint_root,
    run_name="v1",
    push_to_hub=True,
    repo_id="Panhapich/tuna-ocr",
    hf_token=hf_token,
    hub_private=True,
    auto_batch_size=True,  # OOM-probing auto-tune; no-op on CPU
)